# Notebook 06 Part 1: Handling Missing Values in PySpark

## Lecture 10: Detect, Remove and Fill Missing Values


## 1. What Is a Missing Value?

A missing value means data is not available for a field.

Examples:

| customer_name | city | age |
|---|---|---:|
| Amit | Delhi | 25 |
| Priya | null | 28 |
| Rahul | Mumbai | null |

Here:

```text
Priya's city is missing.
Rahul's age is missing.
```

In PySpark, missing values are commonly represented as:

```text
null
None
```

## 2. Null vs Blank String

These are different:

| Value | Meaning |
|---|---|
| `null` | Value is missing |
| `""` | Empty text |
| `"   "` | Text containing only spaces |
| `"NA"` | A text value, not automatically null |

Important:

```text
PySpark does not automatically treat blank strings or "NA" as null.
```

## 3. Windows Local Setup

Run this cell before creating the SparkSession.

In [2]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

## 4. Create SparkSession

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Handling_Missing_Values") \
    .master("local[1]") \
    .config("spark.python.worker.reuse", "false") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

## 5. Create Sample Data

The sample contains:

```text
null values
blank strings
spaces
the text "NA"
```

In [4]:
data = [
    (1, "Amit", "Delhi", 25, 45000.0),
    (2, "Priya", None, 28, 52000.0),
    (3, "Rahul", "Mumbai", None, 48000.0),
    (4, None, "Pune", 30, None),
    (5, "Sneha", "", 27, 55000.0),
    (6, "Karan", "   ", 35, 60000.0),
    (7, "Meera", "NA", None, None)
]

columns = ["customer_id", "customer_name", "city", "age", "salary"]

df = spark.createDataFrame(data, columns)

df.show()

+-----------+-------------+------+----+-------+
|customer_id|customer_name|  city| age| salary|
+-----------+-------------+------+----+-------+
|          1|         Amit| Delhi|  25|45000.0|
|          2|        Priya|  NULL|  28|52000.0|
|          3|        Rahul|Mumbai|NULL|48000.0|
|          4|         NULL|  Pune|  30|   NULL|
|          5|        Sneha|      |  27|55000.0|
|          6|        Karan|      |  35|60000.0|
|          7|        Meera|    NA|NULL|   NULL|
+-----------+-------------+------+----+-------+



## 6. Check Schema

In [5]:
df.printSchema()

root
 |-- customer_id: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: long (nullable = true)
 |-- salary: double (nullable = true)



## 7. Detect Null Values with `isNull()`

`isNull()` returns rows where a column contains null.

In [6]:
df.filter(df.city.isNull()).show()

+-----------+-------------+----+---+-------+
|customer_id|customer_name|city|age| salary|
+-----------+-------------+----+---+-------+
|          2|        Priya|NULL| 28|52000.0|
+-----------+-------------+----+---+-------+



In [7]:
df.filter(df.city.isNull()).count() # count the number of null values

1

## 8. Detect Non-Null Values with `isNotNull()`

`isNotNull()` returns rows where the column contains a value.

In [8]:
df.filter(df.city.isNotNull()).show()

+-----------+-------------+------+----+-------+
|customer_id|customer_name|  city| age| salary|
+-----------+-------------+------+----+-------+
|          1|         Amit| Delhi|  25|45000.0|
|          3|        Rahul|Mumbai|NULL|48000.0|
|          4|         NULL|  Pune|  30|   NULL|
|          5|        Sneha|      |  27|55000.0|
|          6|        Karan|      |  35|60000.0|
|          7|        Meera|    NA|NULL|   NULL|
+-----------+-------------+------+----+-------+



### How This Code Works

```python
col("city").isNull()
```

Checks whether each city value is null.

```python
.cast("int")
```

Converts:

```text
True  → 1
False → 0
```

Then `sum()` adds the ones.

## 11. Null value count in every column

The same task can be written step by step.

In [ ]:
from pyspark.sql.functions import col, lit
for column_name in df.columns:
    null_count = df.filter(col(column_name).isNull()).count()
    print(column_name, ":", null_count)

The loop version is easier to understand.

The single-query version is usually more efficient for larger data.

## 12. Remove Rows Containing Any Null Value

Use:

```python
df.dropna()
```

This removes a row if any selected column contains null.

In [ ]:
df_drop_any = df.dropna()

df_drop_any.show()

Equivalent syntax:

```python
df.na.drop()
```

## 13. Remove Rows Where All Values Are Null

Use:

```python
dropna(how="all")
```

The row is removed only when all checked columns are null.

In [ ]:
df_drop_all = df.dropna(how="all")

df_drop_all.show()

## 14. Remove Nulls from Selected Columns

Suppose a row must have both customer name and city.

Use `subset`.

In [ ]:
df_required_columns = df.dropna(
    subset=["customer_name", "city"]
)

df_required_columns.show()

## 15. Use `thresh` to Keep Rows with Enough Non-Null Values

```python
thresh=4
```

means keep rows having at least four non-null values.

In [ ]:
df_minimum_values = df.dropna(thresh=4)

df_minimum_values.show()

## 16. Fill Missing Values with One Value

For numeric columns:

In [ ]:
df_fill_numeric = df.fillna(0)

df_fill_numeric.show()

PySpark fills only compatible columns.

For example:

```text
0 can fill numeric columns.
It will not replace null text values.
```

## 17. Fill Missing Text Values

In [ ]:
df_fill_text = df.fillna("Unknown")

df_fill_text.show()

`"Unknown"` fills null values only in string columns.

## 18. Fill Different Values in Different Columns

This is the most useful method.

In [ ]:
df_filled = df.fillna({
    "customer_name": "Unknown",
    "city": "Unknown",
    "age": 0,
    "salary": 0.0
})

df_filled.show()

Equivalent syntax:

```python
df.na.fill({...})
```

## 19. Fill Missing Numeric Values with Average

Sometimes filling numeric nulls with `0` is not appropriate.

We can calculate the average and use it for imputation.

In [19]:
from pyspark.sql.functions import avg

average_age = df.select(avg("age")).first()[0]

print("Average age:", average_age)

Average age: 29.0


Fill the missing ages using the calculated average.

In [ ]:
df_age_filled = df.fillna({
    "age": average_age
})

df_age_filled.show()

## 20. Handle Blank Strings

A blank string is not null.

Therefore, this will not detect blank cities:

```python
df.filter(col("city").isNull())
```

We must check blank strings separately.

In [ ]:
df.filter(col("city") == "").show()

## 21. Handle Strings Containing Only Spaces

Use `trim()` to remove spaces before checking.

In [ ]:
from pyspark.sql.functions import trim

df.filter(trim(col("city")) == "").show()

This detects both:

```text
""
"   "
```

## 22. Convert Blank Strings to Null

We can convert blank strings into actual null values.

In [ ]:
from pyspark.sql.functions import when

df_clean_city = df.withColumn(
    "city",
    when(
        trim(col("city")) == "",
        None
    ).otherwise(col("city"))
)

df_clean_city.show()

## 23. Convert `"NA"` to Null

`"NA"` is text, so we must convert it manually.

In [ ]:
df_clean_city = df_clean_city.withColumn(
    "city",
    when(
        col("city") == "NA",
        None
    ).otherwise(col("city"))
)

df_clean_city.show()

## 24. Combine Blank and `"NA"` Handling

A cleaner version:

In [ ]:
df_cleaned = df.withColumn(
    "city",
    when(
        col("city").isNull() |
        (trim(col("city")) == "") |
        (col("city") == "NA"),
        None
    ).otherwise(col("city"))
)

df_cleaned.show()

## 26. Replace Specific Values with `replace()`

Use `replace()` when a particular value must be changed.

Example:

```text
"NA" → "Unknown"
```

In [ ]:
df_replaced = df.replace(
    {"NA": "Unknown"},
    subset=["city"]
    
)

df_replaced.show()

In [20]:
spark.stop()